In [ ]:
#!pip3 install torch torchvision matplotlib seaborn scikit-learn pillow tqdm

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torchvision.models import (
    VGG16_Weights,
    ResNet50_Weights,
    MobileNet_V2_Weights,
    EfficientNet_B0_Weights
)

from torch.utils.data import DataLoader
from tqdm import tqdm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = r"C:\Users\sksaa\OneDrive\Desktop\smartVision AI\smartvision_dataset\classification"

BATCH_SIZE = 32
NUM_CLASSES = 26
EPOCHS = 25

os.makedirs("models", exist_ok=True)

print("Device:", device)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.RandomResizedCrop(224, scale=(0.5,1.0)),  # 🔥 key improvement
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),

    transforms.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.3
    ),

    transforms.ToTensor(),

    transforms.RandomErasing(p=0.3),  # 🔥 simulate black regions

    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [ ]:
train_data = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
val_data   = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=val_transform)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=BATCH_SIZE)

print(train_data.classes)

In [ ]:
def train_model(model, name):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.0001,
        weight_decay=1e-4   # 🔥 FIX OVERFITTING
    )

    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    best_acc = 0

    for epoch in range(EPOCHS):
        model.train()
        correct, total = 0, 0

        for imgs, labels in tqdm(train_loader):
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()   # ✅ correct order

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        scheduler.step()   # ✅ AFTER optimizer

        train_acc = correct / total

        # VALIDATION
        model.eval()
        correct, total = 0, 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_acc = correct / total

        print(f"{name} Epoch {epoch+1}: Train={train_acc:.3f}, Val={val_acc:.3f}")

        # SAVE BEST
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f"models/{name}.pth")

        # 🔥 EARLY STOPPING
        '''epoch > 5 and val_acc < best_acc:
            print("Early stopping")
            break'''

    return model

In [ ]:
from torchvision.models import vgg16, VGG16_Weights

vgg = vgg16(weights=VGG16_Weights.DEFAULT)

# freeze base (PROJECT)
for param in vgg.features.parameters():
    param.requires_grad = False

# classifier
vgg.classifier = nn.Sequential(
    nn.Linear(25088, 512),
    nn.ReLU(),
    nn.BatchNorm1d(512),
    nn.Dropout(0.5),
    nn.Linear(512, NUM_CLASSES)
)

vgg = vgg.to(device)

train_model(vgg, "VGG16")

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

resnet = resnet50(weights=ResNet50_Weights.DEFAULT)

# freeze all
for param in resnet.parameters():
    param.requires_grad = False

# unfreeze last 20 layers (PROJECT)
for param in list(resnet.parameters())[-20:]:
    param.requires_grad = True

# classifier
resnet.fc = nn.Sequential(
    nn.Linear(resnet.fc.in_features, 512),
    nn.ReLU(),
    nn.BatchNorm1d(512),
    nn.Dropout(0.5),
    nn.Linear(512, NUM_CLASSES)
)

resnet = resnet.to(device)

train_model(resnet, "ResNet50")

In [ ]:
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

mobilenet = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)

# freeze base
for param in mobilenet.features.parameters():
    param.requires_grad = False

# classifier
mobilenet.classifier = nn.Sequential(
    nn.BatchNorm1d(mobilenet.last_channel),
    nn.Dropout(0.3),
    nn.Linear(mobilenet.last_channel, NUM_CLASSES)
)

mobilenet = mobilenet.to(device)

train_model(mobilenet, "MobileNetV2")

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

efficient = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

# ✅ Fine-tune (PROJECT allows)
for param in efficient.features.parameters():
    param.requires_grad = True

# ✅ Better classifier
efficient.classifier = nn.Sequential(
    nn.BatchNorm1d(efficient.classifier[1].in_features),
    nn.Dropout(0.5),
    nn.Linear(efficient.classifier[1].in_features, NUM_CLASSES)
)

efficient = efficient.to(device)

train_model(efficient, "EfficientNetB0")